In [ ]:
import pandas as pd
import numpy as np
import re

# ──────────────────────────────────────────────
# FUNGSI KONVERSI USIA
# ──────────────────────────────────────────────

def konversi_usia_baru(usia_str):
    """
    Konversi format 'X Tahun - Y Bulan - Z Hari' ke total bulan.
    Contoh: '1 Tahun - 7 Bulan - 1 Hari' -> 19
            '0 Tahun - 10 Bulan - 23 Hari' -> 10
    Hari diabaikan, hanya ambil tahun + bulan.
    """
    if pd.isna(usia_str):
        return np.nan
    usia_str = str(usia_str).strip()
    try:
        # Ekstrak angka tahun, bulan, hari
        angka = re.findall(r'\d+', usia_str)
        if len(angka) >= 2:
            tahun = int(angka[0])
            bulan = int(angka[1])
            return (tahun * 12) + bulan
        return np.nan
    except:
        return np.nan


# ──────────────────────────────────────────────
# FASE 1: LOAD DATA BARU (EXCEL)
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 1: LOAD DATA BARU")
print("=" * 55)

df_baru = pd.read_excel('../data/raw/dataset_baru.xlsx', header=1)
print(f"✅ Data baru dimuat  : {len(df_baru)} baris")
print(f"   Kolom            : {df_baru.columns.tolist()}\n")

# ──────────────────────────────────────────────
# FASE 2: BERSIHKAN DATA BARU
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 2: BERSIHKAN DATA BARU")
print("=" * 55)

# Drop kolom tidak diperlukan
drop_cols = ['No', 'Tanggal Pengukuran',
             'LiLA', 'BB/U', 'TB/U', 'BB/TB', ]
df_baru.drop(columns=drop_cols, inplace=True)
print(f"✅ Kolom di-drop: {drop_cols}")

# Konversi usia ke bulan
df_baru['Usia_Bulan'] = df_baru['Usia Saat Ukur'].apply(konversi_usia_baru)
df_baru.drop(columns=['Usia Saat Ukur'], inplace=True)
print(f"✅ Usia dikonversi ke bulan")

# Bersihkan JK (strip spasi)
df_baru['JK'] = df_baru['JK'].str.strip()

# Rename kolom agar sesuai dataset lama
df_baru.rename(columns={
    'ZS BB/U' : 'ZS BB/U',
    'ZS TB/U' : 'ZS TB/U',
    'ZS BB/TB': 'ZS BB/TB',
}, inplace=True)

# Susun ulang kolom agar sesuai dataset lama
# Format dataset lama: JK, Usia (dalam format teks), Berat, Tinggi, ZS BB/U, ZS TB/U, ZS BB/TB
# Kita simpan dalam format baru yang sudah dalam bulan
df_baru = df_baru[['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB']]

print(f"✅ Kolom setelah bersih: {df_baru.columns.tolist()}")
print(f"\nSample data baru:")
print(df_baru.head(3).to_string())
print()


# ──────────────────────────────────────────────
# FASE 3: LOAD DATASET LAMA
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 3: LOAD DATASET LAMA")
print("=" * 55)

df_lama = pd.read_csv('../data/raw/dataset.csv')
print(f"✅ Dataset lama dimuat: {len(df_lama)} baris")
print(f"   Kolom: {df_lama.columns.tolist()}\n")

# Konversi usia dataset lama ke bulan juga (jika belum)
def konversi_usia_lama(usia_str):
    """Konversi format '3 tahun' / '15 bulan' ke bulan."""
    if pd.isna(usia_str):
        return np.nan
    usia_str = str(usia_str).strip().lower()
    if 'tahun' in usia_str:
        try:
            return int(usia_str.replace('tahun', '').strip()) * 12
        except:
            return np.nan
    elif 'bulan' in usia_str:
        try:
            return int(usia_str.replace('bulan', '').strip())
        except:
            return np.nan
    return np.nan

if 'Usia' in df_lama.columns:
    df_lama['Usia_Bulan'] = df_lama['Usia'].apply(konversi_usia_lama)
    df_lama.drop(columns=['Usia'], inplace=True)
# Jika sudah 'Usia_Bulan', tidak perlu konversi lagi

# Bersihkan JK dataset lama
df_lama['JK'] = df_lama['JK'].str.strip()

# Drop kolom tidak diperlukan dari dataset lama
drop_lama = ['Nama', 'Tgl Lahir', 'Nama Ortu', 'Desa/Kel',
             'RT', 'RW', 'LiLA', 'Status Gizi', 'Usia']
df_lama.drop(columns=[c for c in drop_lama if c in df_lama.columns], inplace=True)

print(f"✅ Dataset lama setelah bersih: {len(df_lama)} baris")
print(f"   Kolom: {df_lama.columns.tolist()}\n")


# ──────────────────────────────────────────────
# FASE 4: GABUNGKAN DATASET
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 4: GABUNGKAN DATASET")
print("=" * 55)

df_gabung = pd.concat([df_lama, df_baru], ignore_index=True)
print(f"✅ Dataset lama  : {len(df_lama)} baris")
print(f"✅ Dataset baru  : {len(df_baru)} baris")
print(f"✅ Total gabungan: {len(df_gabung)} baris\n")


# ──────────────────────────────────────────────
# FASE 5: CEK DUPLIKAT
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 5: CEK DUPLIKAT")
print("=" * 55)

duplikat = df_gabung.duplicated().sum()
print(f"   Duplikat ditemukan: {duplikat}")
if duplikat > 0:
    df_gabung.drop_duplicates(inplace=True)
    print(f"✅ Duplikat dihapus, sisa: {len(df_gabung)} baris")
else:
    print(f"✅ Tidak ada duplikat")
print()


# ──────────────────────────────────────────────
# FASE 6: CEK MISSING VALUES
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 6: CEK MISSING VALUES")
print("=" * 55)

print("Missing values di dataset gabungan:")
print(df_gabung.isnull().sum().to_string())
print()


# ──────────────────────────────────────────────
# FASE 7: SIMPAN DATASET GABUNGAN
# ──────────────────────────────────────────────
print("=" * 55)
print("FASE 7: SIMPAN DATASET GABUNGAN")
print("=" * 55)

# Simpan sebagai dataset_gabungan.csv
# CATATAN: dataset ini sudah dalam format Usia_Bulan (integer)
# Sehingga data_preparation.py perlu sedikit penyesuaian
df_gabung.to_csv('../data/raw/dataset_gabungan.csv', index=False)
print(f"✅ dataset_gabungan.csv tersimpan!")
print(f"   Berisi {len(df_gabung)} baris dan {len(df_gabung.columns)} kolom")
print(f"   Kolom: {df_gabung.columns.tolist()}")
print()

print("=" * 55)
print("✅ MERGE SELESAI")
print("=" * 55)
print(f"   Selanjutnya:")
print(f"   1. Ganti nama dataset_gabungan.csv -> dataset.csv")
print(f"   2. Update data_preparation.py (lihat catatan di bawah)")
print(f"   3. Jalankan ulang data_preparation.py")
print(f"   4. Jalankan ulang training_rf.py")
print("=" * 55)
print()
print("⚠️  CATATAN PENTING untuk data_preparation.py:")
print("   Dataset gabungan sudah dalam format Usia_Bulan (integer),")
print("   bukan lagi format teks '3 tahun' / '15 bulan'.")
print("   Hapus/skip Fase 3 (konversi usia) di data_preparation.py")
print("   karena kolom 'Usia_Bulan' sudah siap pakai langsung.")



FASE 1: LOAD DATA BARU
✅ Data baru dimuat  : 1397 baris
   Kolom            : ['No', 'Nama', 'JK', 'Alamat', 'Usia Saat Ukur', 'Tanggal Pengukuran', 'Berat', 'Tinggi', 'Cara Ukur', 'LiLA', 'BB/U', 'ZS BB/U', 'TB/U', 'ZS TB/U', 'BB/TB', 'ZS BB/TB']

FASE 2: BERSIHKAN DATA BARU
✅ Kolom di-drop: ['No', 'Tanggal Pengukuran', 'LiLA', 'BB/U', 'TB/U', 'BB/TB']
✅ Usia dikonversi ke bulan
✅ Kolom setelah bersih: ['JK', 'Usia_Bulan', 'Berat', 'Tinggi', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB']

Sample data baru:
  JK  Usia_Bulan  Berat  Tinggi  ZS BB/U  ZS TB/U  ZS BB/TB
0  L          13    9.6    76.0    -0.36    -0.61     -0.13
1  P          19    9.7    76.5    -0.60    -1.76      0.32
2  L          14    8.7    74.5    -1.34    -1.42     -0.95

FASE 3: LOAD DATASET LAMA
✅ Dataset lama dimuat: 1351 baris
   Kolom: ['Nama', 'JK', 'Usia', 'Tgl Lahir', 'Nama Ortu', 'Desa/Kel', 'RT', 'RW', 'Berat', 'Tinggi', 'LiLA', 'ZS BB/U', 'ZS TB/U', 'ZS BB/TB', 'Status Gizi']

✅ Dataset lama setelah bersih: 1351 bar